In [1]:
!pip install fredapi pandas numpy python-dotenv -q

In [2]:
from google.colab import userdata
import os

FRED_API_KEY = userdata.get('FRED_API_KEY')
os.environ['FRED_API_KEY'] = FRED_API_KEY
print("FRED API key loaded:", FRED_API_KEY[:4] + "..." + FRED_API_KEY[-4:])

FRED API key loaded: 596a...0b1c


In [3]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key=FRED_API_KEY)

'''
Core indicators for Level 1 (Minimum) - covers employment, inflation,
rates, production, and the recession target
'''

indicators = {
    'UNRATE':    'Unemployment Rate',
    'CPIAUCSL':  'CPI (All Items)',
    'INDPRO':    'Industrial Production Index',
    'FEDFUNDS':  'Federal Funds Rate',
    'DGS10':     '10-Year Treasury Yield',
    'DGS2':      '2-Year Treasury Yield',
    'GDPC1':     'Real GDP',
    'USREC':     'NBER Recession Indicator (target)',
}

raw_data = {}
for series_id, description in indicators.items():
    print(f"Fetching {series_id} ({description})...")
    raw_data[series_id] = fred.get_series(series_id)

print("\nDone. Series fetched:", list(raw_data.keys()))

Fetching UNRATE (Unemployment Rate)...
Fetching CPIAUCSL (CPI (All Items))...
Fetching INDPRO (Industrial Production Index)...
Fetching FEDFUNDS (Federal Funds Rate)...
Fetching DGS10 (10-Year Treasury Yield)...
Fetching DGS2 (2-Year Treasury Yield)...
Fetching GDPC1 (Real GDP)...
Fetching USREC (NBER Recession Indicator (target))...

Done. Series fetched: ['UNRATE', 'CPIAUCSL', 'INDPRO', 'FEDFUNDS', 'DGS10', 'DGS2', 'GDPC1', 'USREC']


In [4]:
# Build a DataFrame from all series (indexed by date)
df = pd.DataFrame(raw_data)

# Resample everything to monthly frequency:
# - Daily rate series (yields) -> monthly average
# - Quarterly GDP -> forward-filled to monthly
# - Monthly series stay as-is, but we align all to month-start

df_monthly = pd.DataFrame(index=pd.date_range(df.index.min(), df.index.max(), freq='MS'))

for col in df.columns:
    series = df[col].dropna()
    if col in ['DGS10', 'DGS2']:
        # daily -> monthly average
        monthly = series.resample('MS').mean()
    elif col == 'GDPC1':
        # quarterly -> forward fill to monthly
        monthly = series.resample('MS').ffill()
    else:
        monthly = series.resample('MS').mean()  # already monthly, this just aligns index
    df_monthly[col] = monthly

df_monthly = df_monthly.rename(columns=indicators)
df_monthly.index.name = 'Date'

print(df_monthly.shape)
df_monthly.tail(10)

(2062, 8)


,Unemployment Rate,CPI (All Items),Industrial Production Index,Federal Funds Rate,10-Year Treasury Yield,2-Year Treasury Yield,Real GDP,NBER Recession Indicator (target)
Date,,,,,,,,
2025-12-01,4.4,326.031,101.4941,3.72,4.143182,3.500909,24055.749,0.0
2026-01-01,4.3,326.588,101.0388,3.64,4.213500,3.537000,24180.419,0.0
2026-02-01,4.4,327.460,101.9053,3.64,4.125789,3.471579,24180.419,0.0
2026-03-01,4.3,330.293,101.7543,3.64,4.245909,3.714545,24180.419,0.0
2026-04-01,4.3,332.407,102.5198,3.64,4.319091,3.799091,24269.613,0.0
2026-05-01,4.3,333.979,102.5099,3.63,4.484000,3.995000,NaN,0.0
2026-06-01,4.2,332.568,102.7868,3.63,4.470476,4.112857,NaN,0.0
2026-07-01,4.1,332.813,102.9939,3.63,4.599545,4.222727,NaN,0.0
2026-08-01,4.1,NaN,NaN,3.63,4.684286,4.216190,NaN,0.0


In [5]:
# Drop rows before all series have started reporting
df_clean = df_monthly.dropna(how='all')

# Forward-fill small gaps (e.g. a lag in data availability), then drop any remaining leading NaNs
df_clean = df_clean.ffill()
df_clean = df_clean.dropna()

# --- Derived indicators ---
df_clean['Yield Curve Spread'] = df_clean['10-Year Treasury Yield'] - df_clean['2-Year Treasury Yield']
df_clean['Inflation Rate'] = df_clean['CPI (All Items)'].pct_change(periods=12) * 100  # YoY %
df_clean['Unemployment Change'] = df_clean['Unemployment Rate'].diff()
df_clean['Industrial Production Growth'] = df_clean['Industrial Production Index'].pct_change(periods=12) * 100

# Recession target as clean int (0/1)
df_clean['Recession'] = df_clean['NBER Recession Indicator (target)'].round().astype(int)

# Drop rows where derived indicators are NaN (e.g. first 12 months for YoY calcs)
df_final = df_clean.dropna()

print(df_final.shape)
df_final.tail()

(592, 13)


,Unemployment Rate,CPI (All Items),Industrial Production Index,Federal Funds Rate,10-Year Treasury Yield,2-Year Treasury Yield,Real GDP,NBER Recession Indicator (target),Yield Curve Spread,Inflation Rate,Unemployment Change,Industrial Production Growth,Recession
Date,,,,,,,,,,,,,
2026-05-01,4.3,333.979,102.5099,3.63,4.484000,3.995000,24269.613,0.0,0.489000,4.166615,0.0,1.529631,0
2026-06-01,4.2,332.568,102.7868,3.63,4.470476,4.112857,24269.613,0.0,0.357619,3.463531,-0.1,1.289239,0
2026-07-01,4.1,332.813,102.9939,3.63,4.599545,4.222727,24269.613,0.0,0.376818,3.303856,-0.1,1.079455,0
2026-08-01,4.1,332.813,102.9939,3.63,4.684286,4.216190,24269.613,0.0,0.468095,2.945334,0.0,1.347310,0
2026-09-01,4.1,332.813,102.9939,3.63,4.783333,4.373333,24269.613,0.0,0.410000,2.642446,0.0,1.304147,0


In [6]:
# Check for duplicates
print("Duplicate rows:", df_final.index.duplicated().sum())

# Check date range and recession balance
print("Date range:", df_final.index.min(), "to", df_final.index.max())
print("\nRecession label balance:")
print(df_final['Recession'].value_counts())

Duplicate rows: 0
Date range: 1977-06-01 00:00:00 to 2026-09-01 00:00:00

Recession label balance:
Recession
0    534
1     58
Name: count, dtype: int64


In [7]:
import os
os.makedirs('data/processed', exist_ok=True)
df_final.to_csv('data/processed/economic_dataset.csv')
print("Saved to data/processed/economic_dataset.csv")

Saved to data/processed/economic_dataset.csv


In [8]:
# Check the last 6 months for suspicious repeated values (sign of forward-fill on unpublished data)
df_final.tail(6)

,Unemployment Rate,CPI (All Items),Industrial Production Index,Federal Funds Rate,10-Year Treasury Yield,2-Year Treasury Yield,Real GDP,NBER Recession Indicator (target),Yield Curve Spread,Inflation Rate,Unemployment Change,Industrial Production Growth,Recession
Date,,,,,,,,,,,,,
2026-04-01,4.3,332.407,102.5198,3.64,4.319091,3.799091,24269.613,0.0,0.520000,3.779246,0.0,1.376376,0
2026-05-01,4.3,333.979,102.5099,3.63,4.484000,3.995000,24269.613,0.0,0.489000,4.166615,0.0,1.529631,0
2026-06-01,4.2,332.568,102.7868,3.63,4.470476,4.112857,24269.613,0.0,0.357619,3.463531,-0.1,1.289239,0
2026-07-01,4.1,332.813,102.9939,3.63,4.599545,4.222727,24269.613,0.0,0.376818,3.303856,-0.1,1.079455,0
2026-08-01,4.1,332.813,102.9939,3.63,4.684286,4.216190,24269.613,0.0,0.468095,2.945334,0.0,1.347310,0
2026-09-01,4.1,332.813,102.9939,3.63,4.783333,4.373333,24269.613,0.0,0.410000,2.642446,0.0,1.304147,0


In [9]:
# Check the true last reported date for each raw monthly series (before any ffill)
for series_id in ['UNRATE', 'CPIAUCSL', 'INDPRO', 'FEDFUNDS']:
    last_real_date = raw_data[series_id].dropna().index.max()
    print(f"{series_id}: last real value on {last_real_date.date()}")

UNRATE: last real value on 2026-08-01
CPIAUCSL: last real value on 2026-07-01
INDPRO: last real value on 2026-07-01
FEDFUNDS: last real value on 2026-08-01


In [11]:
# --- Trim dataset to avoid unpublished/partial data artifacts ---
# Some monthly series (e.g. CPI, Industrial Production) report with a lag,
# so ffill() can carry forward stale values for months that haven't been
# published yet. We find the true last-reported date for each core monthly
# series and cut the dataset there, so every row reflects real published data.

safe_cutoff = min(
    raw_data[s].dropna().index.max() for s in ['UNRATE', 'CPIAUCSL', 'INDPRO', 'FEDFUNDS']
)
safe_cutoff = safe_cutoff.replace(day=1)  # normalize to first of month

# Keep only rows up to the safe cutoff
df_final = df_final[df_final.index <= safe_cutoff]

# Overwrite the saved dataset with the trimmed, artifact-free version
df_final.to_csv('data/processed/economic_dataset.csv')

print("Safe cutoff date:", safe_cutoff)
print("Final shape:", df_final.shape)
df_final.tail(3)

Safe cutoff date: 2026-07-01 00:00:00
Final shape: (590, 13)


,Unemployment Rate,CPI (All Items),Industrial Production Index,Federal Funds Rate,10-Year Treasury Yield,2-Year Treasury Yield,Real GDP,NBER Recession Indicator (target),Yield Curve Spread,Inflation Rate,Unemployment Change,Industrial Production Growth,Recession
Date,,,,,,,,,,,,,
2026-05-01,4.3,333.979,102.5099,3.63,4.484000,3.995000,24269.613,0.0,0.489000,4.166615,0.0,1.529631,0
2026-06-01,4.2,332.568,102.7868,3.63,4.470476,4.112857,24269.613,0.0,0.357619,3.463531,-0.1,1.289239,0
2026-07-01,4.1,332.813,102.9939,3.63,4.599545,4.222727,24269.613,0.0,0.376818,3.303856,-0.1,1.079455,0


✅ FRED data pulled (8 core indicators)
✅ Aligned to monthly frequency
✅ Missing values handled
✅ Derived indicators created (yield spread, inflation rate, unemployment change, IP growth)
✅ Duplicates checked (0)
✅ Publication-lag artifacts trimmed
✅ indicator_dictionary.xlsx generated
✅ Saved to data/processed/economic_dataset.csv

In [12]:
# Mount Google Drive to persist data across notebooks
from google.colab import drive
drive.mount('/content/drive')

# Create a project folder in your Drive
import os
os.makedirs('/content/drive/MyDrive/fred-economic-agent/data/processed', exist_ok=True)

# Save the final dataset there instead of local /content/
df_final.to_csv('/content/drive/MyDrive/fred-economic-agent/data/processed/economic_dataset.csv')
print("Saved to Google Drive")

Mounted at /content/drive
Saved to Google Drive
